# MORPHEUS: RNA-seq preprocessing for Gene Co-expression Graph (GCG)

We perform transcriptomic data preprocessing for the MORPHEUS framework, specifically tailored for graph-based analysis. The goal is to transform raw RNA-seq gene expression data into a structured format suitable for constructing a Gene Co-expression Graph (GCG)

## Objective
To generate a filtered and normalized RNA-seq matrix that preserves gene-level relationships for downstream graph construction and graph neural network (GNN) modeling

## Pipeline Overview

The preprocessing pipeline consists of the following steps:

1. **Cohort Assembly and Patient-Level Splitting**
   - Load patient IDs and LUAD/LUSC labels
   - Perform stratified split into train/validation/test (7:1:2)

2. **Raw RNA-seq Data Loading**
   - Load FPKM-UQ expression values
   - Align patient IDs across datasets

3. **Gene Filtering**
   - Remove genes with >80% zero expression
   - Retain top 25% most variable genes using Median Absolute Deviation (MAD)

4. **Data Transformation and Normalization**
   - Apply log₂(FPKM-UQ + 1) transformation
   - Perform per-sample z-score normalization

5. **Output Generation**
   - Save filtered RNA matrix (N × G)
   - Save gene identifiers
   - Save split-specific datasets (train/val/test)
   - Save preprocessing statistics for reproducibility


## 1. Setup: Paths, Reproducibility, Output Directories

In [25]:
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Working directory (where your notebook lives)
WORKDIR = Path("/home/steps4growth/gmriechi/Lung-Cancer-Subtyping/GraphBased-Analysis")
os.chdir(WORKDIR)
print("CWD:", Path.cwd())

# Raw RNA roots (GDC download structure)
RNA_ROOTS = {
    "LUAD": Path(
        "/home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/01_download/"
        "rna/TCGA-LUAD/Transcriptome_Profiling/Gene_Expression_Quantification"
    ),
    "LUSC": Path(
        "/home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/01_download/"
        "rna/TCGA-LUSC/Transcriptome_Profiling/Gene_Expression_Quantification"
    ),
}

# Labels file
LABELS_PATH = Path("/home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/multiomics_labels.tsv")

# Project root + outputs (new folder to avoid mixing with PCA outputs)
PROJECT_ROOT = Path("/home/steps4growth/gmriechi/Lung-Cancer-Subtyping")
OUT_BASE     = PROJECT_ROOT / "Data" / "02_preprocessed" / "rna_gcg"
SPLITS_DIR   = PROJECT_ROOT / "Data" / "splits"

OUT_BASE.mkdir(parents=True, exist_ok=True)
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

# Sanity checks
for k, p in RNA_ROOTS.items():
    if not p.exists():
        raise FileNotFoundError(f"Missing RNA root for {k}: {p}")
if not LABELS_PATH.exists():
    raise FileNotFoundError(f"Missing labels file: {LABELS_PATH}")

print("Paths OK")
print("OUT_BASE  :", OUT_BASE)
print("SPLITS_DIR:", SPLITS_DIR)

CWD: /home/steps4growth/gmriechi/Lung-Cancer-Subtyping/GraphBased-Analysis
Paths OK
OUT_BASE  : /home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/02_preprocessed/rna_gcg
SPLITS_DIR: /home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/splits


## 2. Stratified Train/Validation/Test Split (7:1:2)

This cell filters the cohort to LUAD/LUSC, encodes labels (LUAD=0, LUSC=1),
creates a patient-level stratified split (7:1:2), and saves split patient IDs for reuse.

In [27]:
from sklearn.model_selection import train_test_split

labels = pd.read_csv(LABELS_PATH, sep="\t")

labels = labels[labels["subtype_simple"].isin(["LUAD", "LUSC"])].copy()

label_map = {"LUAD": 0, "LUSC": 1}
labels["y"] = labels["subtype_simple"].map(label_map).astype(int)

pids = labels["patient_id"].astype(str).values
y    = labels["y"].values

# 7:1:2 stratified split
train_pids, temp_pids, y_train, y_temp = train_test_split(
    pids, y, test_size=0.30, random_state=SEED, stratify=y
)
val_pids, test_pids, y_val, y_test = train_test_split(
    temp_pids, y_temp, test_size=2/3, random_state=SEED, stratify=y_temp
)

print("Split sizes:", {"train": len(train_pids), "val": len(val_pids), "test": len(test_pids)})
print("Class balance:")
print("  train:", dict(zip(*np.unique(y_train, return_counts=True))))
print("  val  :", dict(zip(*np.unique(y_val,   return_counts=True))))
print("  test :", dict(zip(*np.unique(y_test,  return_counts=True))))

np.save(SPLITS_DIR / "train_pids.npy", train_pids)
np.save(SPLITS_DIR / "val_pids.npy",   val_pids)
np.save(SPLITS_DIR / "test_pids.npy",  test_pids)
np.save(SPLITS_DIR / "y_train.npy", y_train)
np.save(SPLITS_DIR / "y_val.npy",   y_val)
np.save(SPLITS_DIR / "y_test.npy",  y_test)

print("Saved splits to:", SPLITS_DIR)

Split sizes: {'train': 581, 'val': 83, 'test': 167}
Class balance:
  train: {np.int64(0): np.int64(321), np.int64(1): np.int64(260)}
  val  : {np.int64(0): np.int64(46), np.int64(1): np.int64(37)}
  test : {np.int64(0): np.int64(92), np.int64(1): np.int64(75)}
Saved splits to: /home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/splits


## 3. Collect RNA Count Files (UUID folders → TSV paths)

This cell scans the LUAD/LUSC RNA directories, finds the augmented STAR gene count TSV in each UUID folder,
and builds a list of available RNA files for downstream mapping.

In [28]:
def collect_rna_files(root: Path):
    uuid_folders = [p for p in root.iterdir() if p.is_dir()]
    files = []
    for folder in uuid_folders:
        hits = list(folder.glob("*.rna_seq.augmented_star_gene_counts.tsv"))
        if len(hits) == 1:
            files.append({"uuid_folder": folder.name, "path": hits[0]})
    return uuid_folders, files

rna_files = []
for cancer, root in RNA_ROOTS.items():
    _, files = collect_rna_files(root)
    for d in files:
        d["cancer"] = cancer
        rna_files.append(d)

print("Total RNA count TSV files found:", len(rna_files))
print("First 3 examples:")
for x in rna_files[:3]:
    print(x["cancer"], x["uuid_folder"], "->", x["path"].name)

Total RNA count TSV files found: 825
First 3 examples:
LUAD ffec28dd-cc44-4057-bcc3-ccc0cd2331c2 -> d21d91e9-6e19-4c5f-aa2e-cdd41d98e542.rna_seq.augmented_star_gene_counts.tsv
LUAD fe20855f-d3ce-4e02-b2dd-3df15eab44bf -> 564e41d6-f9a4-4a6b-bb97-f8e5423b84bd.rna_seq.augmented_star_gene_counts.tsv
LUAD fdcad604-c1b4-40ff-a652-3c1e5fe5f9a1 -> 2301103f-5cbe-4cb6-b5da-0b83b39e4616.rna_seq.augmented_star_gene_counts.tsv


## 4. Load UUID→Patient Mapping and Attach to RNA Files

This cell loads the existing UUID-to-patient mapping file and joins it with the collected RNA file list,
producing a table of (patient_id, cohort, file_path) aligned to the cohort splits.

In [30]:
MAP_PATH = Path("/home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/02_preprocessed/id_maps/file_uuid_to_patient_id.csv")
if not MAP_PATH.exists():
    raise FileNotFoundError(f"Mapping file not found: {MAP_PATH}")

uuid_map = pd.read_csv(MAP_PATH, dtype=str)

uuid_to_patient = dict(
    zip(uuid_map["file_id"].astype(str).str.strip(),
        uuid_map["case_submitter_id"].astype(str).str.strip().str[:12])
)

rna_files_df = pd.DataFrame(rna_files).copy()
rna_files_df["uuid_folder"] = rna_files_df["uuid_folder"].astype(str).str.strip()
rna_files_df["patient_id"] = rna_files_df["uuid_folder"].map(uuid_to_patient)

print("Mapped files:", rna_files_df["patient_id"].notna().sum(), "/", len(rna_files_df))

rna_files_df = rna_files_df.dropna(subset=["patient_id"]).copy()

# restrict to split patients (train/val/test)
split_pids = set(np.concatenate([train_pids, val_pids, test_pids]).astype(str))
rna_files_df = rna_files_df[rna_files_df["patient_id"].isin(split_pids)].copy()

print("After restricting to split patients:", rna_files_df.shape)

out_map_used = OUT_BASE / "rna_files_mapped.tsv"
rna_files_df.to_csv(out_map_used, sep="\t", index=False)
print("Saved mapped RNA file table:", out_map_used)

rna_files_df.head()

Mapped files: 825 / 825
After restricting to split patients: (825, 4)
Saved mapped RNA file table: /home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/02_preprocessed/rna_gcg/rna_files_mapped.tsv


,uuid_folder,path,cancer,patient_id
0,ffec28dd-cc44-4057-bcc3-ccc0cd2331c2,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,LUAD,TCGA-91-6829
1,fe20855f-d3ce-4e02-b2dd-3df15eab44bf,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,LUAD,TCGA-69-8453
2,fdcad604-c1b4-40ff-a652-3c1e5fe5f9a1,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,LUAD,TCGA-50-5055
3,fd52d8a9-e044-4f17-861a-9780752b20ee,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,LUAD,TCGA-62-A46V
4,fc757476-1d31-470b-b648-79608b201bdc,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,LUAD,TCGA-MP-A4TJ


## 5. Build Patient→FilePath Dictionary (Aligned to Train/Val/Test Order)

This cell creates a patient_id → file_path mapping and defines the fixed patient order
(train → val → test) used to build the raw RNA matrix in the next cell.

In [31]:
pid_to_path = dict(zip(rna_files_df["patient_id"].astype(str), rna_files_df["path"].map(Path)))

ordered_pids = np.concatenate([train_pids, val_pids, test_pids]).astype(str)

print("Ordered patients:", len(ordered_pids))
print("Mapped patients available:", len(pid_to_path))
print("Example:", ordered_pids[:5])

Ordered patients: 831
Mapped patients available: 825
Example: ['TCGA-86-8056' 'TCGA-97-A4M6' 'TCGA-33-6737' 'TCGA-60-2709'
 'TCGA-J2-8194']


## 6. Define a Stable RNA TSV Reader 

This cell defines a robust extractor for augmented STAR gene count TSVs:

- skips comment lines starting with `#`
- keeps only ENSG genes (removes STAR summary rows)
- strips Ensembl version suffixes
- prefers `fpkm_uq_unstranded` and falls back to `unstranded`

The function returns a gene-indexed Series for easy alignment across patients.

In [32]:
from pandas.errors import EmptyDataError

def extract_expr_series(path: Path) -> pd.Series:
    """
    Returns Series(index=ENSG_no_version, values=float32).
    Prefers fpkm_uq_unstranded; falls back to unstranded.
    """
    try:
        df = pd.read_csv(path, sep="\t", comment="#", usecols=["gene_id", "fpkm_uq_unstranded"])
        val_col = "fpkm_uq_unstranded"
    except ValueError:
        df = pd.read_csv(path, sep="\t", comment="#", usecols=["gene_id", "unstranded"])
        val_col = "unstranded"
    except EmptyDataError:
        raise ValueError("Empty/malformed file (comment-only or no table).")

    df["gene_id"] = df["gene_id"].astype(str)

    # Keep true genes only (drops __ rows)
    df = df[df["gene_id"].str.startswith("ENSG")].copy()

    # Strip version suffix
    df["gene_id"] = df["gene_id"].str.replace(r"\.\d+$", "", regex=True)

    df[val_col] = pd.to_numeric(df[val_col], errors="coerce").fillna(0).astype(np.float32)

    s = pd.Series(df[val_col].values, index=df["gene_id"].values)

    # If duplicates after stripping, sum
    s = s.groupby(level=0).sum()

    return s

## 7. Build the Raw RNA Expression Matrix `X_raw` (patient × genes)

This cell loads expression for each patient in the fixed order (train → val → test),
aligns all samples to a common gene index, builds `X_raw`, and saves:

- `rna_raw_X.npy`
- `rna_raw_patient_ids.npy`
- `rna_raw_gene_ids.npy`
- `rna_bad_files.tsv` (any skipped samples)

In [34]:
X_rows = []
kept_pids = []
genes_ref = None
bad = []

for pid in ordered_pids:
    p = pid_to_path.get(pid, None)
    if p is None:
        bad.append((pid, "", "No RNA file mapped"))
        continue

    try:
        s = extract_expr_series(p)
    except Exception as e:
        bad.append((pid, p.name, str(e)))
        continue

    if genes_ref is None:
        genes_ref = s.index
    else:
        s = s.reindex(genes_ref).fillna(0).astype(np.float32)

    X_rows.append(s.values)
    kept_pids.append(pid)

X_raw = np.vstack(X_rows).astype(np.float32)
patient_ids = np.array(kept_pids)
gene_ids = np.array(genes_ref)

print("X_raw shape:", X_raw.shape)
print("Patients kept:", len(patient_ids), "out of", len(ordered_pids))
print("Genes (raw):", len(gene_ids))
print("Bad/skipped:", len(bad))

np.save(OUT_BASE / "rna_raw_X.npy", X_raw)
np.save(OUT_BASE / "rna_raw_patient_ids.npy", patient_ids)
np.save(OUT_BASE / "rna_raw_gene_ids.npy", gene_ids)

pd.DataFrame(bad, columns=["patient_id", "filename", "error"]).to_csv(
    OUT_BASE / "rna_bad_files.tsv", sep="\t", index=False
)

print("Saved raw RNA artifacts to:", OUT_BASE)

X_raw shape: (825, 60660)
Patients kept: 825 out of 831
Genes (raw): 60660
Bad/skipped: 6
Saved raw RNA artifacts to: /home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/02_preprocessed/rna_gcg


## 8. RNA Preprocessing (Zero Filter → MAD Filter → log2 → Per-sample Z-score)

This cell performs the RNA preprocessing required for Gene Co-expression Graph (GCG) construction:

1. **Zero-expression filter:** remove genes with zero expression in more than 80% of patients.
2. **Variance filter (MAD):** retain only the top 25% most variable genes based on Median Absolute Deviation (MAD).
3. **Log transform:** apply log2(x + 1) to stabilize variance.
4. **Per-sample z-score:** normalize each patient profile across the retained gene set.

Outputs saved:
- `rna_filtered_log2z.npy`  (final matrix, N × G)
- `gene_ids_filtered.npy`   (filtered gene list)
- `preprocess_stats.json`   (exact counts and thresholds, including exact G)

In [ ]:
def mad(x, axis=0):
    """Median Absolute Deviation (MAD) along axis."""
    med = np.median(x, axis=axis, keepdims=True)
    return np.median(np.abs(x - med), axis=axis)

N, G_raw = X_raw.shape

In [36]:
# 1) Zero-expression filter (>80% zeros removed)
zero_frac = (X_raw == 0).mean(axis=0)  # per-gene
keep_zero = zero_frac <= 0.80

X1 = X_raw[:, keep_zero]
genes1 = gene_ids[keep_zero]

print("After zero filter:")
print("  kept genes:", X1.shape[1], "removed:", G_raw - X1.shape[1])

After zero filter:
  kept genes: 40379 removed: 20281


In [37]:
# 2) MAD filter (top 25%) 
mad_vals = mad(X1, axis=0)  # per-gene MAD
thr = np.quantile(mad_vals, 0.75)      # top 25% => keep MAD >= 75th percentile
keep_mad = mad_vals >= thr

X2 = X1[:, keep_mad]
genes2 = genes1[keep_mad]

print("After MAD filter (top 25%):")
print("  kept genes (G):", X2.shape[1], "removed:", X1.shape[1] - X2.shape[1])

After MAD filter (top 25%):
  kept genes (G): 10095 removed: 30284


In [38]:
# 3) log2(x + 1) 
X3 = np.log2(X2 + 1.0).astype(np.float32)

In [39]:
# 4) Per-sample z-score (across genes) 
mu = X3.mean(axis=1, keepdims=True)
sd = X3.std(axis=1, keepdims=True)
sd[sd == 0] = 1.0  # prevent divide-by-zero
X_final = ((X3 - mu) / sd).astype(np.float32)

print("\nFinal matrix shape (N × G):", X_final.shape)
print("N (patients):", X_final.shape[0])
print("G (genes)   :", X_final.shape[1])


Final matrix shape (N × G): (825, 10095)
N (patients): 825
G (genes)   : 10095


In [40]:
# Save outputs 
np.save(OUT_BASE / "rna_filtered_log2z.npy", X_final)
np.save(OUT_BASE / "gene_ids_filtered.npy", genes2)
np.save(OUT_BASE / "patient_ids_used.npy", patient_ids)

stats = {
    "N_patients": int(X_final.shape[0]),
    "G_raw": int(G_raw),
    "G_after_zero_filter": int(X1.shape[1]),
    "G_final_after_MAD": int(X_final.shape[1]),
    "zero_filter_threshold": 0.80,
    "mad_quantile_kept": 0.75,
    "mad_threshold_value": float(thr),
    "bad_files_skipped": int(len(pd.read_csv(OUT_BASE / "rna_bad_files.tsv", sep="\t"))),
}

with open(OUT_BASE / "preprocess_stats.json", "w") as f:
    json.dump(stats, f, indent=2)

print("\nSaved:")
print(" -", OUT_BASE / "rna_filtered_log2z.npy")
print(" -", OUT_BASE / "gene_ids_filtered.npy")
print(" -", OUT_BASE / "preprocess_stats.json")



Saved:
 - /home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/02_preprocessed/rna_gcg/rna_filtered_log2z.npy
 - /home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/02_preprocessed/rna_gcg/gene_ids_filtered.npy
 - /home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/02_preprocessed/rna_gcg/preprocess_stats.json


## 9. Construct the Gene Co-expression Graph (GCG) from Training Data Only

We:

- Load the preprocessed RNA matrix (`rna_filtered_log2z.npy`) and the filtered gene list.
- Select **training patients only** (to avoid information leakage).
- Compute the gene–gene Pearson correlation matrix across training patients.
- Build a sparse co-expression graph using a **k-nearest-neighbor (kNN)** rule per gene
  (optionally enforcing a minimum absolute correlation threshold).
- Save the graph objects needed for PyTorch Geometric:
  - `gcg_edge_index.npy`
  - `gcg_edge_weight.npy`
  - `gcg_genes.npy`

In [41]:
# Load preprocessed data
X = np.load(OUT_BASE / "rna_filtered_log2z.npy").astype(np.float32)   # (N × G)
genes = np.load(OUT_BASE / "gene_ids_filtered.npy", allow_pickle=True).astype(str)
pids_used = np.load(OUT_BASE / "patient_ids_used.npy", allow_pickle=True).astype(str)

In [42]:
# Identify which rows are train (since X is in train->val->test order minus skipped)
train_set = set(train_pids.astype(str))
train_mask = np.array([pid in train_set for pid in pids_used], dtype=bool)

X_train = X[train_mask]  # (N_train × G)
print("X_train shape:", X_train.shape)

X_train shape: (578, 10095)


In [43]:
# Correlation across patients: corr(gene_i, gene_j) computed over N_train samples
# We want gene×gene, so correlate columns
C = np.corrcoef(X_train, rowvar=False).astype(np.float32)  # (G × G)
np.fill_diagonal(C, 0.0)

print("Correlation matrix shape:", C.shape)

Correlation matrix shape: (10095, 10095)


In [44]:
# ---- Build sparse graph via kNN per gene ----
K = 20          # k nearest neighbors per gene (tunable)
MIN_ABS_CORR = 0.30  # optional safety threshold; set to 0.0 to disable

G = C.shape[0]
edge_src = []
edge_dst = []
edge_w   = []

absC = np.abs(C)

for i in range(G):
    # pick top-K by absolute correlation
    nbr_idx = np.argpartition(absC[i], -K)[-K:]
    # enforce threshold and remove zeros
    for j in nbr_idx:
        w = C[i, j]
        if np.abs(w) >= MIN_ABS_CORR and i != j:
            edge_src.append(i)
            edge_dst.append(j)
            edge_w.append(w)

edge_index = np.vstack([edge_src, edge_dst]).astype(np.int64)
edge_weight = np.array(edge_w, dtype=np.float32)

print("Edges:", edge_index.shape[1])
print("Edge weight stats:", float(edge_weight.min()), float(edge_weight.max()))

# Save graph
np.save(OUT_BASE / "gcg_edge_index.npy", edge_index)
np.save(OUT_BASE / "gcg_edge_weight.npy", edge_weight)
np.save(OUT_BASE / "gcg_genes.npy", genes)

Edges: 200307
Edge weight stats: -0.8141300678253174 0.9911984801292419


In [45]:
# Save config for reproducibility
gcg_cfg = {
    "K": int(K),
    "min_abs_corr": float(MIN_ABS_CORR),
    "built_from": "TRAIN_ONLY",
    "n_train_patients_used": int(X_train.shape[0]),
    "n_genes": int(G),
    "n_edges": int(edge_index.shape[1])
}
with open(OUT_BASE / "gcg_config.json", "w") as f:
    json.dump(gcg_cfg, f, indent=2)

print("Saved GCG graph files to:", OUT_BASE)
print(" - gcg_edge_index.npy")
print(" - gcg_edge_weight.npy")
print(" - gcg_genes.npy")
print(" - gcg_config.json")

Saved GCG graph files to: /home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/02_preprocessed/rna_gcg
 - gcg_edge_index.npy
 - gcg_edge_weight.npy
 - gcg_genes.npy
 - gcg_config.json


## 10. Build Patient Graph Objects (PyG) Using the Fixed GCG

We:

- Load the preprocessed RNA matrix (`rna_filtered_log2z.npy`) and the fixed GCG edge list.
- For each patient, create a graph where:
  - **nodes = genes** (|V| = G = 10,095)
  - **node features = that patient’s gene expression vector** (shape: G × 1)
  - **edges = fixed gene co-expression graph** (same for all patients)
  - **edge weights = correlation values**
- Construct separate datasets for train/val/test using the saved split IDs.
- Save the resulting PyG-ready arrays to disk for the modeling notebook.

In [46]:
import numpy as np

# Load processed matrix + ids
X = np.load(OUT_BASE / "rna_filtered_log2z.npy").astype(np.float32)      # (N_used × G)
pids_used = np.load(OUT_BASE / "patient_ids_used.npy", allow_pickle=True).astype(str)

# Load labels from splits (original y arrays correspond to original split order)
train_pids = np.load(SPLITS_DIR / "train_pids.npy", allow_pickle=True).astype(str)
val_pids   = np.load(SPLITS_DIR / "val_pids.npy", allow_pickle=True).astype(str)
test_pids  = np.load(SPLITS_DIR / "test_pids.npy", allow_pickle=True).astype(str)
y_train_all = np.load(SPLITS_DIR / "y_train.npy")
y_val_all   = np.load(SPLITS_DIR / "y_val.npy")
y_test_all  = np.load(SPLITS_DIR / "y_test.npy")

# Load GCG graph
edge_index = np.load(OUT_BASE / "gcg_edge_index.npy").astype(np.int64)   # (2, E)
edge_weight = np.load(OUT_BASE / "gcg_edge_weight.npy").astype(np.float32)

# Map pid -> row in X
pid_to_row = {pid: i for i, pid in enumerate(pids_used)}
pids_set = set(pids_used)

# Keep only patients that exist in X (since some were skipped earlier)
train_keep = [(pid, y) for pid, y in zip(train_pids, y_train_all) if pid in pids_set]
val_keep   = [(pid, y) for pid, y in zip(val_pids,   y_val_all)   if pid in pids_set]
test_keep  = [(pid, y) for pid, y in zip(test_pids,  y_test_all)  if pid in pids_set]

print("Patients kept:")
print("  train:", len(train_keep), "of", len(train_pids))
print("  val  :", len(val_keep),   "of", len(val_pids))
print("  test :", len(test_keep),  "of", len(test_pids))

def build_split_arrays(pairs):
    p = np.array([pid for pid, _ in pairs], dtype=str)
    y = np.array([int(lbl) for _, lbl in pairs], dtype=np.int64)
    idx = np.array([pid_to_row[pid] for pid in p], dtype=int)
    X_split = X[idx]  # (N_split × G)
    # node features per patient will be (G × 1)
    return p, y, X_split

train_pid_out, y_train, X_train = build_split_arrays(train_keep)
val_pid_out,   y_val,   X_val   = build_split_arrays(val_keep)
test_pid_out,  y_test,  X_test  = build_split_arrays(test_keep)

print("\nShapes:")
print("  X_train:", X_train.shape, "y_train:", y_train.shape)
print("  X_val  :", X_val.shape,   "y_val  :", y_val.shape)
print("  X_test :", X_test.shape,  "y_test :", y_test.shape)

# Save arrays for modeling notebook
np.save(OUT_BASE / "rna_gcg_train_X.npy", X_train)
np.save(OUT_BASE / "rna_gcg_val_X.npy",   X_val)
np.save(OUT_BASE / "rna_gcg_test_X.npy",  X_test)

np.save(OUT_BASE / "rna_gcg_train_y.npy", y_train)
np.save(OUT_BASE / "rna_gcg_val_y.npy",   y_val)
np.save(OUT_BASE / "rna_gcg_test_y.npy",  y_test)

np.save(OUT_BASE / "rna_gcg_train_pids.npy", train_pid_out)
np.save(OUT_BASE / "rna_gcg_val_pids.npy",   val_pid_out)
np.save(OUT_BASE / "rna_gcg_test_pids.npy",  test_pid_out)

# Graph is shared for all patients
np.save(OUT_BASE / "rna_gcg_edge_index.npy", edge_index)
np.save(OUT_BASE / "rna_gcg_edge_weight.npy", edge_weight)

print("\nSaved PyG-ready RNA arrays + shared graph to:", OUT_BASE)

Patients kept:
  train: 578 of 581
  val  : 80 of 83
  test : 167 of 167

Shapes:
  X_train: (578, 10095) y_train: (578,)
  X_val  : (80, 10095) y_val  : (80,)
  X_test : (167, 10095) y_test : (167,)

Saved PyG-ready RNA arrays + shared graph to: /home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/02_preprocessed/rna_gcg


Build Split-Specific RNA Patient Similarity Graphs (GCG-style)

For downstream GAT modeling, we build patient–patient graphs separately for train/val/test using the
split-specific RNA feature matrices (already filtered + log2 + z-scored).

We connect each patient to its k nearest neighbors using cosine similarity (directed kNN), producing:
- edge_index (2, E) in PyTorch Geometric format
- edge_weight (E,) containing cosine similarities

These graphs will be used in parallel with the WSI graphs in the modeling notebook.

In [48]:
import numpy as np
from sklearn.neighbors import NearestNeighbors
from pathlib import Path

OUT_BASE = Path("/home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/02_preprocessed/rna_gcg")

def build_knn_edge_index(X, k=10, metric="cosine", self_loops=False):
    n = X.shape[0]
    if n == 0:
        return np.zeros((2, 0), dtype=np.int64), np.zeros((0,), dtype=np.float32)

    nn = NearestNeighbors(n_neighbors=min(k + 1, n), metric=metric)
    nn.fit(X)
    dists, idxs = nn.kneighbors(X, return_distance=True)

    src, dst, wts = [], [], []
    for i in range(n):
        for j, d in zip(idxs[i], dists[i]):
            if (not self_loops) and (j == i):
                continue
            src.append(i)
            dst.append(int(j))
            wts.append(float(1.0 - d))  # cosine similarity

    edge_index = np.vstack([np.array(src, dtype=np.int64), np.array(dst, dtype=np.int64)])
    edge_weight = np.array(wts, dtype=np.float32)
    return edge_index, edge_weight

K = 10

edge_train, w_train = build_knn_edge_index(X_train, k=K)
edge_val,   w_val   = build_knn_edge_index(X_val,   k=K)
edge_test,  w_test  = build_knn_edge_index(X_test,  k=K)

print("RNA edges (directed kNN):")
print("Train:", edge_train.shape, w_train.shape)
print("Val  :", edge_val.shape,   w_val.shape)
print("Test :", edge_test.shape,  w_test.shape)

np.save(OUT_BASE / f"rna_train_edge_index_k{K}.npy", edge_train)
np.save(OUT_BASE / f"rna_train_edge_weight_k{K}.npy", w_train)

np.save(OUT_BASE / f"rna_val_edge_index_k{K}.npy", edge_val)
np.save(OUT_BASE / f"rna_val_edge_weight_k{K}.npy", w_val)

np.save(OUT_BASE / f"rna_test_edge_index_k{K}.npy", edge_test)
np.save(OUT_BASE / f"rna_test_edge_weight_k{K}.npy", w_test)

print("\nSaved split-specific RNA graphs to:", OUT_BASE)

RNA edges (directed kNN):
Train: (2, 5780) (5780,)
Val  : (2, 800) (800,)
Test : (2, 1670) (1670,)

Saved split-specific RNA graphs to: /home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/02_preprocessed/rna_gcg
